# Pytorch LSTM 공식 예제
https://docs.pytorch.org/tutorials/beginner/nlp/sequence_models_tutorial.html?utm_source=chatgpt.com

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [11]:
lstm = nn.LSTM(
    input_size=3,
    hidden_size=3, # h, c 모두 동일한 사이즈로 잡힘
)
inputs = [torch.zeros(1, 3) for _ in range(5)]

# hidden state initialize
hidden1 = (
    torch.zeros(1, 1, 3),
    torch.zeros(1, 1, 3)
) # h_0, c_0 [num_layers, batch, hidden_size]
out1 = ""
for i in inputs:
    # 순회 한번당 sequence 하나씩 나아가게 하기
    # 끝까지 가면 hidden에 값을 저장하기
    # seq length, batch, inputsize로 만들기
    out1, hidden1 = lstm(i.view(1, 1, -1), hidden1)
    # out1 = [sequence, batch, 3]
    # hidden = ([num_layers, batch, hidden_size], [num_layers, batch, hidden_size])

print("out1:", out1)
print("hidden1:", hidden1)

inputs = torch.cat(inputs).view(len(inputs), 1, -1)
hidden2 = (torch.zeros(1, 1, 3), torch.zeros(1, 1, 3))

out2, hidden2 = lstm(inputs, hidden2)
print("out2:", out2)
print("hidden2:", hidden2)

out1: tensor([[[-0.2301, -0.0398, -0.1162]]], grad_fn=<MkldnnRnnLayerBackward0>)
hidden1: (tensor([[[-0.2301, -0.0398, -0.1162]]], grad_fn=<StackBackward0>), tensor([[[-0.6446, -0.0941, -0.2772]]], grad_fn=<StackBackward0>))
out2: tensor([[[-0.0916, -0.0248, -0.0481]],

        [[-0.1536, -0.0358, -0.0787]],

        [[-0.1923, -0.0398, -0.0976]],

        [[-0.2158, -0.0405, -0.1092]],

        [[-0.2301, -0.0398, -0.1162]]], grad_fn=<MkldnnRnnLayerBackward0>)
hidden2: (tensor([[[-0.2301, -0.0398, -0.1162]]], grad_fn=<StackBackward0>), tensor([[[-0.6446, -0.0941, -0.2772]]], grad_fn=<StackBackward0>))


In [14]:
def prepare_sequence(seq, to_idx):
    idxs = [to_idx[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

training_data = [
    ("The dog ate the apple".split(), ["DET", "NN", "V", "DET", "NN"]),
    ("Everybody read that book".split(), ["NN", "V", "DET", "NN"])
]

word_to_ix = {}

for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

print(word_to_ix)
tag_to_ix = {
    "DET": 0,
    "NN": 1,
    "V": 2
}

EMBEDDING_DIM = 6
HIDDEN_DIM = 6

{'The': 0, 'dog': 1, 'ate': 2, 'the': 3, 'apple': 4, 'Everybody': 5, 'read': 6, 'that': 7, 'book': 8}


In [15]:
class LSTMTagger(nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        # [T, ]
        embeds = self.word_embeddings(sentence)
        # [T, embedidng_dim]
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        # [S, B, hidden_size] -> [S, B*hidden_size]
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        # [S, tagset_size]
        tag_scores = F.log_softmax(tag_space, dim=1)
        # [S, tagset_size]

        return tag_scores

In [19]:
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

print(tag_to_ix)
print(training_data[0][1])

# See what the scores are before training
# Note that element i,j of the output is the score for tag j for word i.
# Here we don't need to train, so the code is wrapped in torch.no_grad()
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    print(torch.argmax(tag_scores, dim=1))

for epoch in range(300):  # again, normally you would NOT do 300 epochs, it is toy data
    for sentence, tags in training_data:
        # Step 1. Remember that Pytorch accumulates gradients.
        # We need to clear them out before each instance
        model.zero_grad()

        # Step 2. Get our inputs ready for the network, that is, turn them into
        # Tensors of word indices.
        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        # Step 3. Run our forward pass.
        tag_scores = model(sentence_in)

        # Step 4. Compute the loss, gradients, and update the parameters by
        #  calling optimizer.step()
        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()

# See what the scores are after training
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)

    # The sentence is "the dog ate the apple".  i,j corresponds to score for tag j
    # for word i. The predicted tag is the maximum scoring tag.
    # Here, we can see the predicted sequence below is 0 1 2 0 1
    # since 0 is index of the maximum value of row 1,
    # 1 is the index of maximum value of row 2, etc.
    # Which is DET NOUN VERB DET NOUN, the correct sequence!
    print(torch.argmax(tag_scores, dim=1))

{'DET': 0, 'NN': 1, 'V': 2}
['DET', 'NN', 'V', 'DET', 'NN']
tensor([2, 2, 2, 2, 2])
tensor([0, 1, 2, 0, 1])


In [23]:
sum([p.numel() for p in model.lstm.parameters()])

336

In [27]:
for n, p in model.lstm.named_parameters():
    print(n, p.numel(), p.shape)

weight_ih_l0 144 torch.Size([24, 6])
weight_hh_l0 144 torch.Size([24, 6])
bias_ih_l0 24 torch.Size([24])
bias_hh_l0 24 torch.Size([24])
